# 01 Quickstart: train DeepVoid on a small sample

This notebook runs the whole pipeline on a small sample that ships with the repository, so you can see every step work before pointing it at your own simulation:

1. put density grids into a project folder,
2. compute the labels (the T-web classification) from the matter density,
3. train a small U-Net to recover those labels from sparse tracers,
4. score it on a part of the box it never trained on, and look at the result.

Use a GPU runtime (Runtime → Change runtime type → GPU).

**What the sample is.** The IllustrisTNG TNG300-3-Dark simulation at redshift 0 (a 205 h⁻¹ Mpc box), gridded to 128³ cells of 1.6 h⁻¹ Mpc: the dark-matter density, and the density of the most massive subhalos at a mean separation of 7 h⁻¹ Mpc. See `sample/README.md` for credits.

**What this is not.** A 128³ grid is too coarse for the DeepVoid label convention (smoothing of 1 h⁻¹ Mpc needs cells smaller than that), and the model here is small and trains for minutes. The scores at the end show that the pipeline works. They are not comparable with the scores in the repository README: the labels here are smoothed over 2 h⁻¹ Mpc instead of 1, which removes the thinnest walls and filaments, and with them much of what makes the task hard, so these scores can come out as high as the README's or higher.

In [ ]:
# Get the code. While the repository is private this needs a Colab secret
# named GITHUB_TOKEN; once it is public the try-block does nothing.
url = "https://github.com/skumagai3/deepvoid.git"
try:
    from google.colab import userdata
    url = url.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
!rm -rf /content/deepvoid && git clone -q {url} /content/deepvoid
!pip -q install /content/deepvoid matplotlib
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. A project folder

A `Project` is a folder with your grids (`data/`), a catalog that names them (`catalog.yaml`) and your training runs (`runs/`). Grids are arrays of shape (n, n, n), axis order (x, y, z), covering a periodic box; lengths are in h⁻¹ Mpc.

In [ ]:
import numpy as np
from deepvoid import Project

sample = np.load("/content/deepvoid/sample/tng300-3-dark_n128.npz")
box_size = float(sample["box_size"])          # 205 h^-1 Mpc

project = Project("/content/quickstart")
project.add_density("dm", sample["dm"], box_size=box_size, tracer="dm")
project.add_density("subhalos/L7", sample["subhalos_L7"], box_size=box_size,
                    tracer="subhalos", spacing=7.0)

## 2. Labels

`add_labels` computes the T-web classification from the matter density: smooth the density, solve for the gravitational potential, take the tidal tensor (the matrix of second derivatives of the potential) and count its eigenvalues above a threshold in each cell. Zero is a void, one a wall, two a filament, three a halo.

Two numbers define the labels: the smoothing scale `sigma_eff` and the eigenvalue `threshold`. DeepVoid uses 1 h⁻¹ Mpc and 0.65. This sample's cells are 1.6 h⁻¹ Mpc, larger than 1, so here we use `sigma_eff=2.0`. With your own data on a finer grid, keep the default.

In [ ]:
labels = project.add_labels("labels", density="dm", sigma_eff=2.0)
print(project.summary())

In [ ]:
from deepvoid import figures

fig = figures.fields_figure({"dark matter": project.load("dm").data,
                             "subhalos, spacing 7": project.load("subhalos/L7").data},
                            labels.data, index=64, box_size=box_size)

## 3. The run's settings

`project.config` returns every setting of a run, filled in with defaults. The ones changed here are the ones a small 128³ box forces: thinner slabs, smaller crops and a smaller network. `configs/template.yaml` in the repository explains every setting, with the values used for the full-size model.

- **Split.** The box is cut into train, validation and test slabs along x, with buffers between them. The model trains on crops from the training slab, the validation slab picks the best epoch, and the test slab is scored once at the end.
- **Smoothing.** 93% of the cells of the subhalo grid are empty. A second input channel holds the same field smoothed with a Gaussian of half the tracer spacing.
- **Class weights.** Halos are about 0.5% of the cells; `auto` makes rare classes count for more in the loss.

In [ ]:
cfg = project.config(
    "quickstart",
    inputs=["subhalos/L7"],
    labels="labels",
    data={"split": {"fractions": [0.5, 0.25, 0.25], "buffer": 8},
          "smoothing": {"sigma_spacing": 0.5}},
    model={"depth": 3},
    training={"crop": 48, "val_crop": 24, "epochs": 10,
              "loss": {"class_weights": "auto"},
              "early_stopping": {"monitor": "val_mcc", "patience": 5}},
    evaluate={"crop": 64, "margin": 16},
)

import yaml
print(yaml.safe_dump(cfg, sort_keys=False))

## 4. Train

Each epoch is 200 steps of 8 random crops, followed by a pass over fixed validation crops. The line printed per epoch gives the validation scores; `val_mcc` is the Matthews correlation coefficient over the four classes (1 is perfect, 0 is chance). Everything is saved in the run folder as it goes, and rerunning the cell after an interruption continues where it stopped.

In [ ]:
from deepvoid.train import run

out = run(cfg)
print("run folder:", out)

## 5. Results

`Run` opens a finished run. The test scores were written at the end of training, from a prediction of the whole box scored on the test slab only.

In [ ]:
from deepvoid.runs import Run

r = Run(out)
for name, s in r.scores().items():
    print(f"{name}: four-class MCC {s['mcc']:.3f}, void MCC {s['void_mcc']:.3f}, void F1 {s['void_f1']:.3f}")

fig = figures.training_curves([r], keys=("val_mcc", "val_void_mcc", "val_loss"))

In [ ]:
scores = r.predict()                      # class scores for every cell, shape (n, n, n, 4)
test = r.regions["test"]                  # the slab the model never trained on
fig = figures.prediction_figure(r.input(), r.labels(), scores, index=(test.start + test.stop) // 2,
                                box_size=box_size, input_label="subhalos (smoothed channel)")

## Next

- `02_your_simulation.ipynb` does the same with your own density and tracers.
- `configs/template.yaml` lists every setting and what it does.
- `docs/training.md` covers training on several tracer spacings at once, resuming, seeds and logs.